<a href="https://colab.research.google.com/github/yichungcheng/SSL_MSM_Endoscope_Spine_surgery/blob/main/00_SSLMSM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Self-supervised Learning for Endoscopic Video Analysis

In [1]:
# 1. 設置與安裝函式庫
!pip install -q torch torchvision
!pip install -q segmentation_models_pytorch
!pip install -q albumentations # 數據增強庫
!pip install -q pycocotools # 處理 COCO 格式的官方工具，用於解析多邊形
!pip install tensorflow

# 2. 匯入必要的函式庫
import os
import json
import numpy as np
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from pycocotools.coco import COCO
from google.colab import drive
import shutil

# 3. 連接 Google Drive
drive.mount('/content/gdrive')

# 4. 定義路徑
# 您的資料夾名稱
DRIVE_DIR = '/content/gdrive/MyDrive/PMBM/論文程式/MSM'
# DATASET_NAME = 'Endoscope Spine Surgery.v12i.coco'
# UNZIP_PATH = '/content/dataset'


DATA_ROOT = DRIVE_DIR # 假設數據已經在 Drive 資料夾內

# 檢查路徑結構
print(f"數據根目錄: {DATA_ROOT}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 2.3 MB/s eta 0:00:00
Mounted at /content/gdrive
數據根目錄: /content/gdrive/MyDrive/PMBM/論文程式/MSM


##### ssl_msn/
##### ├── main.py
##### ├── dataset.py
##### ├── augment.py
##### ├── model.py
##### ├── loss.py
##### └── train.py

dataset.py

In [ ]:
# import tensorflow as tf
# import os

# IMG_SIZE = 224

# def parse_image(path):
#     img = tf.io.read_file(path)
#     img = tf.image.decode_jpeg(img, channels=3)
#     img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
#     img = tf.cast(img, tf.float32) / 255.0
#     return img

# def build_dataset(frames_root, batch_size):
#     """
#     frames_root/
#       ├── Video01/*.jpg
#       ├── Video02/*.jpg
#       └── ...
#     """

#     all_image_paths = []

#     # Iterate through each video directory from Video01 to Video80
#     for i in range(1, 81):
#         video_dir = os.path.join(frames_root, f"Video{i:02d}")
#         if not os.path.isdir(video_dir):
#             print(f"Warning: Video directory not found: {video_dir}")
#             continue

#         # List all image files within the current video directory
#         for filename in os.listdir(video_dir):
#             if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
#                 all_image_paths.append(os.path.join(video_dir, filename))

#     print(f"[INFO] Total frames loaded: {len(all_image_paths)}")

#     ds = tf.data.Dataset.from_tensor_slices(all_image_paths)
#     ds = ds.shuffle(buffer_size=2048)

#     ds = ds.map(parse_image, num_parallel_calls=tf.data.AUTOTUNE)
#     ds = ds.batch(batch_size)
#     ds = ds.prefetch(tf.data.AUTOTUNE)

#     return ds

三、Strong Augmentation（MSN 的靈魂）
augment.py四、MSN View Generator（same image → 2 views）

In [ ]:
# import tensorflow as tf
# def strong_augment(x):
#     x = tf.image.random_flip_left_right(x)
#     x = tf.image.random_brightness(x, 0.4)
#     x = tf.image.random_contrast(x, 0.6, 1.4)
#     x = tf.image.random_saturation(x, 0.6, 1.4)

#     # random crop
#     crop_size = tf.random.uniform([], 180, 224, dtype=tf.int32)
#     x = tf.image.random_crop(x, size=[crop_size, crop_size, 3])
#     x = tf.image.resize(x, (224, 224))

#     return x
# def two_views(batch):
#     v1 = tf.map_fn(strong_augment, batch)
#     v2 = tf.map_fn(strong_augment, batch)
#     return v1, v2


五、Backbone + Projection Head
model.py

In [ ]:
# !pip install tensorflow
# import tensorflow as tf
# from tensorflow.keras import layers, models
# def build_backbone():
#     base = tf.keras.applications.ResNet50(
#         include_top=False,
#         weights=None,
#         input_shape=(224, 224, 3),
#         pooling="avg"
#     )
#     return base
# def projection_head(dim=256):
#     return models.Sequential([
#         layers.Dense(1024, activation="gelu"),
#         layers.Dense(dim)
#     ])


六、MSN Loss（Student → Teacher distribution）
loss.py

In [ ]:
# import tensorflow as tf
# def msn_loss(student, teacher, temperature=0.1):
#     student = tf.nn.log_softmax(student / temperature, axis=-1)
#     teacher = tf.nn.softmax(teacher / temperature, axis=-1)
#     return -tf.reduce_mean(tf.reduce_sum(teacher * student, axis=-1))


七、EMA Teacher Update（核心）

In [ ]:
# def update_teacher(student, teacher, momentum=0.996):
#     for s, t in zip(student.trainable_variables,
#                     teacher.trainable_variables):
#         t.assign(momentum * t + (1.0 - momentum) * s)


八、Training Step（tf.function）
train.py

In [ ]:
# This cell is now redundant as train_step is defined in train.py and imported by the main script.

九、Main（完整可跑）

In [ ]:
import tensorflow as tf
import os
import sys
import importlib

DRIVE_DIR = '/content/gdrive/MyDrive/PMBM/論文程式/MSM'
MODULE_DIR = os.path.join(DRIVE_DIR, 'ssl_msn')
os.makedirs(MODULE_DIR, exist_ok=True)
MODEL_SAVE_DIR = os.path.join(MODULE_DIR, 'models')
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

# Add MODULE_DIR to Python path to enable imports and give it priority
if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR) # Use insert(0) for higher priority
    print(f"Added {MODULE_DIR} to sys.path: {sys.path}")
else:
    print(f"{MODULE_DIR} already in sys.path: {sys.path}")

# Write dataset.py
with open(os.path.join(MODULE_DIR, 'dataset.py'), 'w') as f:
    f.write('''import tensorflow as tf
import os

IMG_SIZE = 224

def parse_image(path):
    tf.print(f"Parsing image path: {path}, dtype: {path.dtype}") # Debug print
    img = tf.io.read_file(path)
    # 讀取jpg
    # img = tf.image.decode_jpeg(img, channels=3)
    # 讀取png
    img = tf.image.decode_png(img, channels=3) # Changed to decode_png
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32) / 255.0
    return img

def build_dataset(frames_root, batch_size, split_type=None):
    """
    frames_root/
      ├── Video01/*.jpg
      ├── Video02/*.jpg
      └── ...
    """

    split_ranges = {
        'train': range(1, 2),
        'validation': range(2, 49),
        'test': range(49, 81)
    }

    all_image_paths = []

    if split_type and split_type in split_ranges:
        video_indices = split_ranges[split_type]
    else: # Default to all videos if split_type is not specified or invalid
        video_indices = range(1, 81)

    # Iterate through each video directory based on the split_type
    for i in video_indices:
        video_dir = os.path.join(frames_root, f"video{i:02d}")
        if not os.path.isdir(video_dir):
            print(f"Warning: video directory not found: {video_dir}")
            continue

        # List all image files within the current video directory
        for filename in os.listdir(video_dir):
            if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                all_image_paths.append(os.path.join(video_dir, filename))

    print(f"[INFO] Total frames loaded for {split_type or 'all'}: {len(all_image_paths)}")
    print(f"all_image_paths content before creating dataset: {all_image_paths[:5]} (first 5) and type: {type(all_image_paths[0]) if all_image_paths else 'empty'}")

    if not all_image_paths:
        print("No image paths found. Returning an empty dataset.")
        return tf.data.Dataset.from_tensor_slices(tf.constant([], dtype=tf.string))

    ds = tf.data.Dataset.from_tensor_slices(all_image_paths)
    ds = ds.shuffle(buffer_size=2048)

    ds = ds.map(parse_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds
''')

# Write augment.py
with open(os.path.join(MODULE_DIR, 'augment.py'), 'w') as f:
    f.write('''import tensorflow as tf
def strong_augment(x):
    x = tf.image.random_flip_left_right(x)
    x = tf.image.random_brightness(x, 0.4)
    x = tf.image.random_contrast(x, 0.6, 1.4)
    x = tf.image.random_saturation(x, 0.6, 1.4)

    # random crop
    crop_size = tf.random.uniform([], 180, 224, dtype=tf.int32)
    x = tf.image.random_crop(x, size=[crop_size, crop_size, 3])
    x = tf.image.resize(x, (224, 224))

    return x
def two_views(batch):
    v1 = tf.map_fn(strong_augment, batch)
    v2 = tf.map_fn(strong_augment, batch)
    return v1, v2
''')

# Write model.py
with open(os.path.join(MODULE_DIR, 'model.py'), 'w') as f:
    f.write('''import tensorflow as tf
from tensorflow.keras import layers, models
def build_backbone():
    base = tf.keras.applications.ResNet50(
        include_top=False,
        weights=None,
        input_shape=(224, 224, 3),
        pooling="avg"
    )
    return base
def projection_head(dim=256):
    return models.Sequential([
        layers.Dense(1024, activation="gelu"),
        layers.Dense(dim)
    ])
''')

# Write loss.py
with open(os.path.join(MODULE_DIR, 'loss.py'), 'w') as f:
    f.write('''import tensorflow as tf
def msn_loss(student, teacher, temperature=0.1):
    student = tf.nn.log_softmax(student / temperature, axis=-1)
    teacher = tf.nn.softmax(teacher / temperature, axis=-1)
    return -tf.reduce_mean(tf.reduce_sum(teacher * student, axis=-1))
''')

# Write train.py - This needs to include update_teacher as well, and import msn_loss
with open(os.path.join(MODULE_DIR, 'train.py'), 'w') as f:
    f.write('''import tensorflow as tf
from loss import msn_loss # Now we can import it from the local file

def update_teacher(student, teacher, momentum=0.996):
    for s, t in zip(student.trainable_variables,
                    teacher.trainable_variables):
        t.assign(momentum * t + (1.0 - momentum) * s)

@tf.function
def train_step(view1, view2,
               student_backbone, teacher_backbone,
               student_head, teacher_head,
               optimizer):

    with tf.GradientTape() as tape:
        s_feat = student_backbone(view1, training=True)
        s_proj = student_head(s_feat, training=True)

        t_feat = teacher_backbone(view2, training=False)
        t_proj = teacher_head(t_feat, training=False)

        loss = msn_loss(s_proj, tf.stop_gradient(t_proj))

    vars = student_backbone.trainable_variables + student_head.trainable_variables
    grads = tape.gradient(loss, vars)
    optimizer.apply_gradients(zip(grads, vars))

    update_teacher(student_backbone, teacher_backbone)
    update_teacher(student_head, teacher_head)

    return loss
''')

print(f"Created module files in {MODULE_DIR} and added to Python path.")


# Now import from the created modules using import module and then module.func
import dataset
import augment
import model
import train

# Explicitly reload modules to ensure the latest version is used after writing files
importlib.reload(dataset)
importlib.reload(augment)
importlib.reload(model)
importlib.reload(train)

config_path = f'{DRIVE_DIR}/cholec80/config.json'
# Revert IMAGE_DIR to the previous value to verify its structure with ls
IMAGE_DIR = f"{DATA_ROOT}/cholec80/cholec80_extracted/frames"
BATCH_SIZE = 16
EPOCHS = 2

print(f"Checking contents of DATA_ROOT/cholec80/cholec80_extracted: {DRIVE_DIR}/cholec80/cholec80_extracted/")
!ls -F "{DRIVE_DIR}/cholec80/cholec80_extracted/" # Inspect parent directory

print(f"Checking contents of IMAGE_DIR: {IMAGE_DIR}")
!ls -F "{IMAGE_DIR}" # Inspect IMAGE_DIR

# Call functions using the imported module prefix
train_dataset = dataset.build_dataset(IMAGE_DIR, BATCH_SIZE, split_type='train')

student_backbone = model.build_backbone()
teacher_backbone = model.build_backbone()
student_head = model.projection_head()
teacher_head = model.projection_head()

# initialize teacher
for s, t in zip(student_backbone.variables,
                    teacher_backbone.variables):
    t.assign(s)
for s, t in zip(student_head.variables, teacher_head.variables):
    t.assign(s)

optimizer = tf.keras.optimizers.Adam(1e-4)
print("training start")
for epoch in range(EPOCHS):
    total_loss = 0.0
    steps = 0

    for batch in train_dataset:
        v1, v2 = augment.two_views(batch)
        loss = train.train_step(
            v1, v2,
            student_backbone, teacher_backbone,
            student_head, teacher_head,
            optimizer
        )
        total_loss += loss
        steps += 1

    print(f"Epoch {epoch+1}: loss={total_loss/steps:.4f}")
# 十、Pretrained Backbone 用於 Segmentation
student_backbone.save(f'{MODEL_SAVE_DIR}/msn_pretrained_backbone.keras')

/content/gdrive/MyDrive/PMBM/論文程式/MSM/ssl_msn already in sys.path: ['/content/gdrive/MyDrive/PMBM/論文程式/MSM/ssl_msn', '/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython', '/tmp/tmppzquc2u9']
Created module files in /content/gdrive/MyDrive/PMBM/論文程式/MSM/ssl_msn and added to Python path.
Checking contents of DATA_ROOT/cholec80/cholec80_extracted: /content/gdrive/MyDrive/PMBM/論文程式/MSM/cholec80/cholec80_extracted/
frames/  phase_annotations/  tool_annotations/
Checking contents of IMAGE_DIR: /content/gdrive/MyDrive/PMBM/論文程式/MSM/cholec80/cholec80_extracted/frames
video01/  video11/  video21/  video31/	video41/  video51/  video61/  video71/
video02/  video12/  video22/  video32/	video42/  video52/  video62/  video72/
video03/  video13/  video23/  video33/	video43/  video53/  v

1️⃣ 原始手術影像
2️⃣ CLS Attention Map（最後一層）
3️⃣ Patch Similarity Map（cosine similarity）

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# Main Visualization Function
# ------------------------------------------------------------
def visualize_vit_results(model, image):

    """
    model: ViT encoder (must return tokens + attention)
    image: single image tensor (H, W, 3), normalized 0~1
    """

    # Expand batch
    input_tensor = tf.expand_dims(image, 0)

    # Forward
    tokens, attn = model(input_tensor, return_attention=True)

    tokens = tokens[0]           # (N, D)
    attn = attn[0]               # (heads, N, N)

    # --------------------------------------------------------
    # 1️⃣ Original Image
    # --------------------------------------------------------
    original_img = image.numpy()

    # --------------------------------------------------------
    # 2️⃣ Attention Map (CLS → patches)
    # --------------------------------------------------------
    # Average over heads
    attn_mean = tf.reduce_mean(attn, axis=0)  # (N, N)

    cls_attn = attn_mean[0, 1:]  # CLS token attention to patches

    num_patches = int(np.sqrt(len(cls_attn)))
    cls_attn_map = tf.reshape(cls_attn, (num_patches, num_patches))

    cls_attn_map = cls_attn_map.numpy()

    # Normalize for visualization
    cls_attn_map = (cls_attn_map - cls_attn_map.min()) / \
                   (cls_attn_map.max() - cls_attn_map.min() + 1e-8)

    # Resize to image size
    cls_attn_map = tf.image.resize(
        cls_attn_map[..., None],
        (original_img.shape[0], original_img.shape[1])
    ).numpy().squeeze()

    # --------------------------------------------------------
    # 3️⃣ Similarity Map
    # --------------------------------------------------------
    # Remove CLS token
    patch_tokens = tokens[1:]  # (N-1, D)

    # Choose reference patch (center patch)
    ref_idx = len(patch_tokens) // 2
    ref_token = patch_tokens[ref_idx]

    # Cosine similarity
    patch_tokens_norm = tf.math.l2_normalize(patch_tokens, axis=1)
    ref_token_norm = tf.math.l2_normalize(ref_token, axis=0)

    similarity = tf.matmul(
        patch_tokens_norm,
        tf.expand_dims(ref_token_norm, -1)
    )

    similarity = tf.squeeze(similarity)

    similarity_map = tf.reshape(
        similarity,
        (num_patches, num_patches)
    ).numpy()

    # Normalize
    similarity_map = (similarity_map - similarity_map.min()) / \
                     (similarity_map.max() - similarity_map.min() + 1e-8)

    similarity_map = tf.image.resize(
        similarity_map[..., None],
        (original_img.shape[0], original_img.shape[1])
    ).numpy().squeeze()

    # --------------------------------------------------------
    # Plot 3 images side-by-side
    # --------------------------------------------------------
    plt.figure(figsize=(18, 6))

    # 1️⃣ Original
    plt.subplot(1, 3, 1)
    plt.imshow(original_img)
    plt.title("Original Endoscopic Frame")
    plt.axis("off")

    # 2️⃣ Attention
    plt.subplot(1, 3, 2)
    plt.imshow(original_img)
    plt.imshow(cls_attn_map, cmap="jet", alpha=0.5)
    plt.title("CLS Attention Map")
    plt.axis("off")

    # 3️⃣ Similarity
    plt.subplot(1, 3, 3)
    plt.imshow(original_img)
    plt.imshow(similarity_map, cmap="jet", alpha=0.5)
    plt.title("Patch Similarity Map")
    plt.axis("off")

    plt.tight_layout()
    plt.show()



# 假設你已經有:
# model = pretrained_vit_encoder
# image = dataset.take(1) 取出的單張圖
image= f"{DATA_ROOT}/cholec80/cholec80_extracted/frames/video01/video01_000002.png"
model = f'{MODEL_SAVE_DIR}/msn_pretrained_backbone.keras'
visualize_vit_results(model, image)


TypeError: 'str' object is not callable